In [3]:
from datasets import load_dataset

dataset = load_dataset("Wangrongsheng/ag_news")

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

## Step 2: Clean and Normalize the Text

This step involves lowercasing all strings and stripping out special characters, punctuation, or numbers. This reduces vocabulary noise for the model.

In [4]:
import re

def clean_text(examples):
    # Lowercase the text
    examples['text'] = [text.lower() for text in examples['text']]
    # Remove special characters, punctuation, and numbers
    examples['text'] = [re.sub(r'[^a-zA-Z\s]', '', text) for text in examples['text']]
    return examples

# Apply the cleaning function to the dataset
# Using batched=True for efficiency if the dataset is large
cleaned_dataset = dataset.map(clean_text, batched=True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [5]:
# Display an example of the cleaned text
print(cleaned_dataset['train'][0]['text'])
print(dataset['train'][0]['text'])

wall st bears claw back into the black reuters reuters  shortsellers wall streets dwindlingband of ultracynics are seeing green again
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.


## Step 3: Tokenize the Strings into Numbers

This step creates a vocabulary lookup dictionary using a Tokenizer, assigning a unique integer ID to every unique word. Neural networks cannot interpret characters or letters; they only understand numbers and matrices.

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Define the maximum number of words to keep, based on word frequency
VOCAB_SIZE = 10000

# Initialize the Tokenizer
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<unk>")

# Fit the tokenizer on the training data (cleaned text)
tokenizer.fit_on_texts(cleaned_dataset['train']['text'])

# Convert text to sequences of integers
train_sequences = tokenizer.texts_to_sequences(cleaned_dataset['train']['text'])
test_sequences = tokenizer.texts_to_sequences(cleaned_dataset['test']['text'])

print(f"Example of original text: {cleaned_dataset['train'][0]['text']}")
print(f"Example of tokenized sequence: {train_sequences[0]}")
print(f"Vocabulary size: {len(tokenizer.word_index)}")

Example of original text: wall st bears claw back into the black reuters reuters  shortsellers wall streets dwindlingband of ultracynics are seeing green again
Example of tokenized sequence: [392, 325, 1526, 1, 100, 55, 2, 813, 24, 24, 1, 392, 1989, 1, 5, 1, 35, 3894, 738, 296]
Vocabulary size: 91344


## Step 4: Pad and Truncate Sequences

This step uses `pad_sequences` to cap text lengths to a uniform size. Shorter articles get trailing zeros; longer articles are trimmed down. Recurrent neural networks require the input tensor dimensions to be consistently shaped across the entire training batch.

In [7]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Define the maximum sequence length
MAX_SEQUENCE_LENGTH = 50

# Pad and truncate the sequences
padded_train_sequences = pad_sequences(train_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
padded_test_sequences = pad_sequences(test_sequences, maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')

print(f"Original length of first training sequence: {len(train_sequences[0])}")
print(f"Padded/truncated length of first training sequence: {len(padded_train_sequences[0])}")
print(f"Example of padded/truncated sequence: {padded_train_sequences[0]}")

Original length of first training sequence: 20
Padded/truncated length of first training sequence: 50
Example of padded/truncated sequence: [ 392  325 1526    1  100   55    2  813   24   24    1  392 1989    1
    5    1   35 3894  738  296    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0]


## Step 5: Convert Labels to Categorical Vectors

This step one-hot encodes the target variable labels. For instance, a class category label of 2 becomes `[0, 0, 1, 0]`. This transforms the classification targets into a probability distribution format that directly matches what the model's terminal output layer will predict.

In [8]:
from tensorflow.keras.utils import to_categorical

# Get the labels
train_labels = cleaned_dataset['train']['label']
test_labels = cleaned_dataset['test']['label']

# Determine the number of unique classes
num_classes = len(set(train_labels))

# One-hot encode the labels
encoded_train_labels = to_categorical(train_labels, num_classes=num_classes)
encoded_test_labels = to_categorical(test_labels, num_classes=num_classes)

print(f"Original label for first training example: {train_labels[0]}")
print(f"One-hot encoded label for first training example: {encoded_train_labels[0]}")
print(f"Number of classes: {num_classes}")

Original label for first training example: 2
One-hot encoded label for first training example: [0. 0. 1. 0.]
Number of classes: 4


## Step 6: Define the RNN Model Architecture

This step involves stacking three specific layers sequentially:
- An Embedding Layer to translate integer IDs into dense, semantic vector spaces.
- A SimpleRNN Layer to track sequential memory across the phrase.
- A Dense Output Layer using a softmax activation to output classification scores.

This establishes the pathway through which the text features are processed and compressed into predictions.

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

# Define embedding dimensions
EMBEDDING_DIM = 100

# Build the RNN model
model = Sequential([
    Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH),
    SimpleRNN(units=32, return_sequences=False),
    Dense(num_classes, activation='softmax')
])

# Display the model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Step 7: Compile and Train the Model

This step involves running `model.compile()` with an optimizer (like `adam`) and a classification loss function, then triggering `model.fit()` using your training data, defined epochs, and a validation split. This starts the active mathematical optimization loop where weights are adjusted iteratively based on prediction errors.

In [10]:
from tensorflow.keras.optimizers import Adam

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Define training parameters
EPOCHS = 10
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2

# Train the model
history = model.fit(
    padded_train_sequences,
    encoded_train_labels,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    verbose=1
)

Epoch 1/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 69s 22ms/step - accuracy: 0.8048 - loss: 0.5521 - val_accuracy: 0.8589 - val_loss: 0.4312
Epoch 2/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 82s 22ms/step - accuracy: 0.8870 - loss: 0.3585 - val_accuracy: 0.8503 - val_loss: 0.4283
Epoch 3/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 84s 23ms/step - accuracy: 0.9053 - loss: 0.3032 - val_accuracy: 0.8627 - val_loss: 0.4538
Epoch 4/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 67s 22ms/step - accuracy: 0.9195 - loss: 0.2568 - val_accuracy: 0.8563 - val_loss: 0.4314
Epoch 5/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 67s 22ms/step - accuracy: 0.9306 - loss: 0.2226 - val_accuracy: 0.8499 - val_loss: 0.5330
Epoch 6/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 67s 22ms/step - accuracy: 0.9389 - loss: 0.1970 - val_accuracy: 0.8561 - val_loss: 0.4915
Epoch 7/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 82s 22ms/step - accuracy: 0.9458 - loss: 0.1748 - val_accuracy: 0.8539 - val_loss: 0.5228
Epoch 8/10
3000/3000 ━━━━━━━━━━━━━━━━━━━━ 68s 23ms/step - accuracy: 0.9507 -

## Step 8: Read and Evaluate the Target Accuracy

This step monitors the logs generated at the end of each epoch or runs a dedicated `model.evaluate()` step on your test subset. The console outputs individual metric readouts for accuracy (how well it fits training data) and `val_accuracy` (how well it performs on unseen validation data) so you can quantify the model's performance.

In [11]:
# Evaluate the model on the test set
loss, accuracy = model.evaluate(padded_test_sequences, encoded_test_labels, verbose=0)

print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Test Loss: 0.5336
Test Accuracy: 0.8675


## Make Predictions with the Trained Model

Now we can use the trained RNN model to predict the category of new, unseen text.

In [12]:
def preprocess_new_text(text, tokenizer, max_sequence_length):
    # 1. Clean the text
    cleaned_text = re.sub(r'[^a-zA-Z\s]', '', text.lower())

    # 2. Tokenize the text
    sequence = tokenizer.texts_to_sequences([cleaned_text])

    # 3. Pad/truncate the sequence
    padded_sequence = pad_sequences(sequence, maxlen=max_sequence_length, padding='post', truncating='post')

    return padded_sequence

# Example new text
new_text = "Google acquires new AI startup specializing in natural language processing."

# Preprocess the new text
processed_input = preprocess_new_text(new_text, tokenizer, MAX_SEQUENCE_LENGTH)

print(f"Original new text: {new_text}")
print(f"Processed input for model: {processed_input}")

Original new text: Google acquires new AI startup specializing in natural language processing.
Processed input for model: [[ 192 4456   17    1 2819    1    6 2166 3173 3727    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0]]


In [13]:
import numpy as np

# Make a prediction
prediction = model.predict(processed_input)

# Get the predicted class (index with the highest probability)
predicted_class_index = np.argmax(prediction)

# Map the class index back to a label (assuming 0: World, 1: Sports, 2: Business, 3: Sci/Tech)
# Note: This mapping is based on common AG News dataset labels. Verify if needed.
class_labels = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}
predicted_label = class_labels.get(predicted_class_index, 'Unknown')

print(f"Prediction probabilities: {prediction[0]}")
print(f"Predicted class index: {predicted_class_index}")
print(f"Predicted category: {predicted_label}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 489ms/step
Prediction probabilities: [6.9215293e-03 2.6868822e-04 1.5392494e-01 8.3888489e-01]
Predicted class index: 3
Predicted category: Sci/Tech
